# Ocean colour - Sentinel-3 OLCI chlorophyll monthly via openEO

The `sentinel-3-olci-chlorophyll-monthly` recipe loads OLCI Level-2 ocean chlorophyll (neural-net retrieval, `CHL_NN`) and reduces it to a **monthly mean** server-side (NetCDF). A starting point for marine productivity and water-quality monitoring. We use a coastal box and one month.

```bash
pip install earthlens[openeo]
```

## Credentials

openEO authenticates with CDSE via OIDC. The download cell below runs for real when credentials are present (env `OPENEO_CLIENT_ID` / `OPENEO_CLIENT_SECRET`, or a cached refresh token from a prior interactive login) and prints a skip note otherwise - so this notebook executes cleanly with or without secrets. See the Authentication page.

In [ ]:
import os
from pathlib import Path


def has_openeo_credentials() -> bool:
    """Whether some openEO OIDC credential source is available."""
    if os.environ.get("OPENEO_CLIENT_ID") and os.environ.get("OPENEO_CLIENT_SECRET"):
        return True
    if os.environ.get("OPENEO_REFRESH_TOKEN"):
        return True
    return (Path.home() / ".openeo").exists()


def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note.

    Keeps the notebook executing top-to-bottom with no errors whether or not
    CDSE OIDC credentials are configured (so the docs build never needs secrets).
    """
    if not has_openeo_credentials():
        print(
            "No openEO OIDC credentials found - skipping the live download.\n"
            "Set OPENEO_CLIENT_ID / OPENEO_CLIENT_SECRET (headless) or sign in once\n"
            "interactively (see the Authentication page) to run this cell for real."
        )
        return []
    paths = facade.download(**download_kwargs)
    for path in paths:
        print(path)
    return paths


## Build the request

In [ ]:
from earthlens import EarthLens

facade = EarthLens(
    data_source='openeo',
    variables={'sentinel-3-olci-chlorophyll-monthly': []},
    start='2023-07-01', end='2023-07-31',
    lat_lim=[36.0, 40.0], lon_lim=[-1.0, 4.0],
    path='data/openeo-chl',
)
g = facade.datasource._resolved['sentinel-3-olci-chlorophyll-monthly']
print('collection:', g.collection_id, '| bands:', g.bands)

## Download (server-side execution)

In [ ]:
paths = run_or_skip(facade)
paths